In [23]:

# --- 6. PRINT VERIFICATION RESULTS ---
print("\n" + "="*45)
print("--- ASSESSMENT DATA RESULTS ---")
print("="*45)
print(f"Records Processed: {valid_records_count}")
print(f"Records in Target Class ({status_class_prefix}xx): {target_class_count}")

print("\n--- Count Per Status Code ---")
for code, count in sorted(status_code_counts.items()):
    print(f"  Status {code}: {count} records")

print("\n--- Five Most Requested URLs (Target Class) ---")
if top_5_urls:
    for idx, (url, count) in enumerate(top_5_urls, 1):
        print(f"  {idx}. {url} ({count} requests)")
else:
    print("  No URLs found for the target class.")

print("\n--- File Verification ---")
print(f"Filtered CSV File: {filtered_csv_path}")
print(f"JSON Summary File: {json_summary_path}")
print(f"Backup Verification: {'Successful (Backup Exists)' if backup_exists else 'Failed'}")
print("="*45)


--- ASSESSMENT DATA RESULTS ---
Records Processed: 1106
Records in Target Class (3xx): 178

--- Count Per Status Code ---
  Status 200: 915 records
  Status 206: 8 records
  Status 302: 157 records
  Status 304: 21 records
  Status 404: 5 records

--- Five Most Requested URLs (Target Class) ---
  1. GET / HTTP/1.1 (53 requests)
  2. POST /process.php HTTP/1.1 (38 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (36 requests)
  4. GET /home.php HTTP/1.1 (9 requests)
  5. POST /action.php HTTP/1.1 (8 requests)

--- File Verification ---
Filtered CSV File: Lab6_File_Handling_Additional_Exercises\output\Lacdan_TUPM-26-8415_filtered_incidents.csv
JSON Summary File: Lab6_File_Handling_Additional_Exercises\output\Lacdan_TUPM-26-8415_costliest_incidents.json
Backup Verification: Successful (Backup Exists)


In [24]:
import csv
import json
import os
from pathlib import Path

# --- 1. STUDENT METADATA & PARAMETERS ---
STUDENT_NAME = "Lacdan"
STUDENT_ID = "TUPM-26-8415"
SEED_DIGIT = 5

# Generation Rules based on your sheet:
# Operating System: Android if SEED_DIGIT is even; iOS if odd (5 is odd -> iOS)
TARGET_OS = "iOS"
# User Behavior Class: (SEED_DIGIT % 5) + 1 -> (5 % 5) + 1 = 1
TARGET_BEHAVIOR_CLASS = 1

# File paths
dataset_path = Path("data/user_behavior_dataset.csv")
output_dir = Path("output")
backup_dir = Path("backup")
output_dir.mkdir(parents=True, exist_ok=True)
backup_dir.mkdir(parents=True, exist_ok=True)

filtered_csv_path = output_dir / f"{STUDENT_NAME}_{STUDENT_ID}_device_filtered.csv"
json_summary_path = output_dir / f"{STUDENT_NAME}_{STUDENT_ID}_device_summary.json"
backup_csv_path = backup_dir / f"{STUDENT_NAME}_{STUDENT_ID}_device_backup.csv"

# --- 2. STREAMING & PROCESSING DATA ---
total_records = 0
os_matches = 0
behavior_matches = 0

total_battery_drain = 0.0
total_data_usage = 0.0
filtered_rows = []

print("Processing mobile device usage dataset...")

if dataset_path.exists():
    with open(dataset_path, mode="r", encoding="utf-8", errors="ignore") as file:
        reader = csv.reader(file)
        header = next(reader, None)  # Skip header

        for row in reader:
            if not row or len(row) < 6:
                continue
            
            total_records += 1
            
            # Assuming typical column structure from the dataset:
            # User ID, Device Model, Operating System, App Usage Time, Screen On Time, Battery Drain, Number of Apps Installed, Data Usage, Age, Gender, User Behavior Class
            # Adjust indices if your CSV columns differ slightly:
            user_id = row[0].strip()
            device_model = row[1].strip()
            os_val = row[2].strip()
            
            try:
                battery_drain = float(row[5].strip())
                data_usage = float(row[7].strip())
                behavior_class = int(row[10].strip())
            except (ValueError, IndexError):
                continue

            # Filter by Operating System
            if os_val.lower() == TARGET_OS.lower():
                os_matches += 1

                # Filter by Target Behavior Class
                if behavior_class == TARGET_BEHAVIOR_CLASS:
                    behavior_matches += 1
                    total_battery_drain += battery_drain
                    total_data_usage += data_usage
                    filtered_rows.append([user_id, device_model, os_val, battery_drain, data_usage, behavior_class])
else:
    print(f"Dataset not found at {dataset_path}. Please place 'user_behavior_dataset.csv' inside the 'data/' folder.")

# Calculate averages safely
avg_battery_drain = (total_battery_drain / behavior_matches) if behavior_matches > 0 else 0.0
avg_data_usage = (total_data_usage / behavior_matches) if behavior_matches > 0 else 0.0

# --- 3. EXPORTING FILTERED CSV ---
with open(filtered_csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)", "User Behavior Class"])
    writer.writerows(filtered_rows)

# Get exported file size
csv_file_size = filtered_csv_path.stat().st_size if filtered_csv_path.exists() else 0

# --- 4. WRITE JSON SUMMARY ---
summary_data = {
    "student_name": STUDENT_NAME,
    "student_id": STUDENT_ID,
    "target_os": TARGET_OS,
    "target_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records_in_dataset": total_records,
    "records_matching_os": os_matches,
    "records_in_target_behavior_class": behavior_matches,
    "average_battery_drain_mah": round(avg_battery_drain, 2),
    "average_data_usage_mb": round(avg_data_usage, 2),
    "exported_csv_size_bytes": csv_file_size
}

with open(json_summary_path, mode="w", encoding="utf-8") as json_file:
    json.dump(summary_data, json_file, indent=4)

# --- 5. CREATE BACKUP ---
if filtered_csv_path.exists():
    with open(filtered_csv_path, mode="r", encoding="utf-8") as src, open(backup_csv_path, mode="w", newline="", encoding="utf-8") as dst:
        dst.write(src.read())
    backup_exists = True
else:
    backup_exists = False

# --- 6. PRINT VERIFICATION RESULTS ---
print("\n" + "="*40)
print("--- ASSESSMENT DATA RESULTS ---")
print("="*40)
print(f"Operating System & Behavior Class: {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print(f"Total Records in Dataset: {total_records}")
print(f"Records Matching OS ({TARGET_OS}): {os_matches}")
print(f"Records in Target Behavior Class: {behavior_matches}")
print(f"Average Battery Drain: {avg_battery_drain:.2f} mAh/day")
print(f"Average Data Usage: {avg_data_usage:.2f} MB/day")

# Added Count Verification matching the exported row count vs target matches
count_verified = (len(filtered_rows) == behavior_matches)
print(f"Count Verification: {'Verified (Match)' if count_verified else 'Mismatch'} ({len(filtered_rows)} exported rows)")

print(f"Exported CSV File: {filtered_csv_path} ({csv_file_size} bytes)")
print(f"JSON Summary File: {json_summary_path}")
print(f"Backup Verification: {'Successful (Backup Exists)' if backup_exists else 'Failed'}")
print("="*40)

Processing mobile device usage dataset...

--- ASSESSMENT DATA RESULTS ---
Operating System & Behavior Class: iOS, Class 1
Total Records in Dataset: 700
Records Matching OS (iOS): 146
Records in Target Behavior Class: 22
Average Battery Drain: 467.23 mAh/day
Average Data Usage: 201.45 MB/day
Count Verification: Verified (Match) (22 exported rows)
Exported CSV File: output\Lacdan_TUPM-26-8415_device_filtered.csv (826 bytes)
JSON Summary File: output\Lacdan_TUPM-26-8415_device_summary.json
Backup Verification: Successful (Backup Exists)


In [2]:
import csv
import json
import os
from datetime import datetime

# --- Configuration & Constants ---
TARGET_YEAR = "2020"
STUDENT_ID = "Lacdan_TUPM-26-8415"
INPUT_CSV = "data/Global_Cybersecurity_Threats_2015-2024.csv"

# Ensure output and backup directories exist
os.makedirs("output", exist_ok=True)
os.makedirs("backup", exist_ok=True)

filtered_csv_path = f"output/{STUDENT_ID}_filtered.csv"
json_summary_path = f"output/{STUDENT_ID}_summary.json"
backup_json_path = f"backup/{STUDENT_ID}_backup.json"
report_txt_path = f"output/{STUDENT_ID}_notes.txt"

def parse_financial_loss(val):
    if not val or str(val).strip() == "":
        return 0.0
    try:
        cleaned = str(val).replace("$", "").replace(",", "").strip()
        return float(cleaned)
    except ValueError:
        return 0.0

total_records_in_dataset = 0
target_incidents = []
attack_type_counts = {}
total_financial_loss = 0.0

# --- Read and Process Dataset ---
with open(INPUT_CSV, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    fieldnames = reader.fieldnames
    
    year_key = next((k for k in fieldnames if 'year' in k.lower() or 'date' in k.lower()), 'Year')
    attack_key = next((k for k in fieldnames if 'attack' in k.lower() or 'type' in k.lower()), 'Attack Type')
    loss_key = next((k for k in fieldnames if 'loss' in k.lower() or 'financial' in k.lower() or 'cost' in k.lower()), None)

    for row in reader:
        total_records_in_dataset += 1
        row_year = str(row.get(year_key, "")).strip()
        
        if TARGET_YEAR in row_year:
            target_incidents.append(row)
            
            # Count attack types
            attack_type = row.get(attack_key, "Unknown").strip()
            attack_type_counts[attack_type] = attack_type_counts.get(attack_type, 0) + 1
            
            # Accumulate financial loss
            if loss_key:
                loss = parse_financial_loss(row.get(loss_key, "0"))
                total_financial_loss += loss

incidents_in_target_year = len(target_incidents)
average_financial_loss = (total_financial_loss / incidents_in_target_year) if incidents_in_target_year > 0 else 0.0

# Sort five costliest incidents descending
sorted_incidents = sorted(
    target_incidents,
    key=lambda x: parse_financial_loss(x.get(loss_key, "0")) if loss_key else 0.0,
    reverse=True
)
five_costliest = sorted_incidents[:5]

# --- 1. Export Filtered CSV ---
with open(filtered_csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(target_incidents)

# --- 2. Create JSON Summary Data Structure ---
summary_data = {
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_records_in_dataset,
    "incidents_in_target_year": incidents_in_target_year,
    "attack_type_counts": attack_type_counts,
    "total_financial_loss_usd_millions": round(total_financial_loss, 2),
    "average_financial_loss_usd_millions": round(average_financial_loss, 2),
    "five_costliest_incidents": five_costliest
}

# Write JSON Summary File
with open(json_summary_path, mode="w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# --- 3. Create Backup Verification JSON File ---
with open(backup_json_path, mode="w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# --- 4. Generate Text Report ---
report_content = f"""==================================================
LABORATORY 6: FILE HANDLING (ADDITIONAL EXERCISES)
Student: {STUDENT_ID}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
==================================================

[ASSESSMENT DATA]
1. Target year: {TARGET_YEAR}
2. Total records in dataset: {total_records_in_dataset:,}
3. Incidents in target year: {incidents_in_target_year:,}
4. Total financial loss (USD millions): ${total_financial_loss:,.2f}M
5. Average financial loss (USD millions): ${average_financial_loss:,.2f}M

[COUNT PER ATTACK TYPE]
"""
for attack, count in attack_type_counts.items():
    report_content += f"  - {attack}: {count}\n"

report_content += "\n[FIVE COSTLIEST INCIDENTS]\n"
for i, inc in enumerate(five_costliest, 1):
    loss_val = parse_financial_loss(inc.get(loss_key, "0")) if loss_key else 0.0
    inc_id = inc.get('Incident ID', inc.get('ID', f'Incident #{i}'))
    report_content += f"  {i}. ID: {inc_id} | Loss: ${loss_val:,.2f}M\n"

report_content += f"""
[FILE REFERENCES]
- Filtered CSV: {filtered_csv_path}
- JSON Summary: {json_summary_path}
- Backup Verification: {backup_json_path}
- Report File: {report_txt_path}
==================================================
"""

with open(report_txt_path, mode="w", encoding="utf-8") as f:
    f.write(report_content)

print("SUCCESS: All assessment metrics, filtered CSV, JSON summary, backup verification, and text reports generated successfully!")
print(report_content)

SUCCESS: All assessment metrics, filtered CSV, JSON summary, backup verification, and text reports generated successfully!
LABORATORY 6: FILE HANDLING (ADDITIONAL EXERCISES)
Student: Lacdan_TUPM-26-8415
Generated: 2026-11-04 20:09:35

[ASSESSMENT DATA]
1. Target year: 2020
2. Total records in dataset: 3,000
3. Incidents in target year: 315
4. Total financial loss (USD millions): $15,767.95M
5. Average financial loss (USD millions): $50.06M

[COUNT PER ATTACK TYPE]
  - Malware: 37
  - Phishing: 61
  - Man-in-the-Middle: 47
  - DDoS: 56
  - SQL Injection: 55
  - Ransomware: 59

[FIVE COSTLIEST INCIDENTS]
  1. ID: Incident #1 | Loss: $99.90M
  2. ID: Incident #2 | Loss: $99.83M
  3. ID: Incident #3 | Loss: $99.81M
  4. ID: Incident #4 | Loss: $99.72M
  5. ID: Incident #5 | Loss: $99.64M

[FILE REFERENCES]
- Filtered CSV: output/Lacdan_TUPM-26-8415_filtered.csv
- JSON Summary: output/Lacdan_TUPM-26-8415_summary.json
- Backup Verification: backup/Lacdan_TUPM-26-8415_backup.json
- Report Fil